In [1]:
%pip install --upgrade ultralytics

Note: you may need to restart the kernel to use updated packages.


Configure paths / Paths

In [2]:
from pathlib import Path
import torch
import ultralytics
from ultralytics import YOLO

PROJECT = Path(r"C:\Projects\smoke_vehicle_detection")
DATASET = PROJECT / "EcoWatch-Final-YOLO"
DATA_YAML = DATASET / "data.yaml"
OUTPUT_ROOT = PROJECT / "runs" / "ecowatch" / "model_comparison"

print("Ultralytics:", ultralytics.__version__)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

if not DATA_YAML.exists():
    raise FileNotFoundError(
        f"Cannot find dataset YAML: {DATA_YAML}\n"
        "Update DATA_YAML above to the actual path of your data.yaml file."
    )

DEVICE = 0 if torch.cuda.is_available() else "cpu"
print("Dataset YAML:", DATA_YAML)
print("Device:", DEVICE)


Ultralytics: 8.4.192
PyTorch: 2.6.0+cu124
CUDA available: True
GPU: NVIDIA GeForce RTX 4060 Laptop GPU
Dataset YAML: C:\Projects\smoke_vehicle_detection\EcoWatch-Final-YOLO\data.yaml
Device: 0


Train YOLO11s

In [3]:
MODEL_LABEL = "YOLO11s"
MODEL_WEIGHTS = "yolo11s.pt"
RUN_NAME = "YOLO11s"

EPOCHS = 100
IMAGE_SIZE = 640
BATCH_SIZE = 8  # Reduce to 4 or 2 if CUDA runs out of memory.
WORKERS = 4
PATIENCE = 20
SEED = 42

# Start from pretrained weights for transfer learning.
# Transfer learning සඳහා pretrained weights වලින් පටන් ගන්න.
model = YOLO(MODEL_WEIGHTS)

train_results = model.train(
    data=str(DATA_YAML),
    epochs=EPOCHS,
    imgsz=IMAGE_SIZE,
    batch=BATCH_SIZE,
    device=DEVICE,
    workers=WORKERS,
    pretrained=True,
    patience=PATIENCE,
    seed=SEED,
    deterministic=True,
    cache=False,
    project=str(OUTPUT_ROOT),
    name=RUN_NAME,
    exist_ok=True,
    plots=True,
    amp=True,
    verbose=True,
)

BEST_WEIGHTS = OUTPUT_ROOT / RUN_NAME / "weights" / "best.pt"
if not BEST_WEIGHTS.exists():
    raise FileNotFoundError(f"Training finished but best.pt was not found: {BEST_WEIGHTS}")

print("Best weights saved at:", BEST_WEIGHTS)


Ultralytics 8.4.192  Python-3.13.5 torch-2.6.0+cu124 CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 8188MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=C:\Projects\smoke_vehicle_detection\EcoWatch-Final-YOLO\data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=YO

Validate the model

In [4]:
# Evaluate on validation split only while choosing the model.
best_model = YOLO(str(BEST_WEIGHTS))
metrics = best_model.val(
    data=str(DATA_YAML),
    split="val",
    imgsz=IMAGE_SIZE,
    batch=BATCH_SIZE,
    device=DEVICE,
    plots=True,
    project=str(OUTPUT_ROOT),
    name=f"{RUN_NAME}_validation",
    exist_ok=True,
)

box = metrics.box
summary = {
    "model": MODEL_LABEL,
    "weights": str(BEST_WEIGHTS),
    "precision": float(box.mp),
    "recall": float(box.mr),
    "mAP50": float(box.map50),
    "mAP50_95": float(box.map),
}
summary


Ultralytics 8.4.192  Python-3.13.5 torch-2.6.0+cu124 CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 8188MiB)
YOLO11s summary (fused): 100 layers, 9,413,187 parameters, 0 gradients, 21.4 GFLOPs
val: Fast image access  (ping: 0.0±0.0 ms, read: 321.0±87.7 MB/s, size: 30.0 KB)
val: Scanning C:\Projects\smoke_vehicle_detection\EcoWatch-Final-YOLO\valid\labels.cache... 283 images, 160 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 283/283 118.7Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 36/36 1.5it/s 23.8s0.1s
                   all        283        135      0.859      0.748      0.807       0.44
Speed: 1.2ms preprocess, 13.2ms inference, 0.0ms loss, 0.5ms postprocess per image
Results saved to C:\Projects\smoke_vehicle_detection\runs\ecowatch\model_comparison\YOLO11s_validation


{'model': 'YOLO11s',
 'weights': 'C:\\Projects\\smoke_vehicle_detection\\runs\\ecowatch\\model_comparison\\YOLO11s\\weights\\best.pt',
 'precision': 0.8585791495219217,
 'recall': 0.7481481481481481,
 'mAP50': 0.8071450274388394,
 'mAP50_95': 0.4404206371508894}

Preview predictions

In [5]:
SAMPLE_IMAGE = DATASET / "test" / "images" / "smoke_100040_jpg.rf.15d894fa79e52871ab7cdfba716370c9.jpg"
if SAMPLE_IMAGE.exists():
    best_model.predict(
        source=str(SAMPLE_IMAGE),
        conf=0.20,
        imgsz=IMAGE_SIZE,
        device=DEVICE,
        save=True,
        project=str(OUTPUT_ROOT),
        name="YOLO11s_preview",
        exist_ok=True,
    )
else:
    print("Update SAMPLE_IMAGE to an existing image path:", SAMPLE_IMAGE)


image 1/1 C:\Projects\smoke_vehicle_detection\EcoWatch-Final-YOLO\test\images\smoke_100040_jpg.rf.15d894fa79e52871ab7cdfba716370c9.jpg: 640x640 3 smokes, 79.9ms
Speed: 5.5ms preprocess, 79.9ms inference, 2.9ms postprocess per image at shape (1, 3, 640, 640)
Results saved to C:\Projects\smoke_vehicle_detection\runs\ecowatch\model_comparison\YOLO11s_preview


In [8]:

from pathlib import Path
from ultralytics import YOLO
import torch

PROJECT = Path(r"C:\Projects\smoke_vehicle_detection")

MODEL_PATH = (
    PROJECT / "runs" / "ecowatch" / "model_comparison"
    / "YOLO11s" / "weights" / "best.pt"
)

# Replace this with the path to your video.
VIDEO_PATH = PROJECT / 'test_video' / "test_video.mp4"

OUTPUT_DIR = PROJECT / "runs" / "ecowatch" / "video_detection"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if not MODEL_PATH.exists():
    raise FileNotFoundError(f"Model not found: {MODEL_PATH}")

if not VIDEO_PATH.exists():
    raise FileNotFoundError(
        f"Video not found: {VIDEO_PATH}\n"
        "Change VIDEO_PATH to your actual video file."
    )

device = 0 if torch.cuda.is_available() else "cpu"
model = YOLO(str(MODEL_PATH))

results = model.predict(
    source=str(VIDEO_PATH),
    conf=0.25,
    imgsz=640,
    device=device,
    save=True,
    project=str(OUTPUT_DIR),
    name="video_result_YOLO11s",
    exist_ok=True,
    stream=True,
    verbose=False
)

frame_count = 0
frames_with_smoke = 0

for result in results:
    frame_count += 1

    if result.boxes is not None and len(result.boxes) > 0:
        frames_with_smoke += 1

print("Video processing completed.")
print("Frames processed:", frame_count)
print("Frames with smoke detections:", frames_with_smoke)
print("Annotated video folder:", OUTPUT_DIR / "video_result")


Results saved to C:\Projects\smoke_vehicle_detection\runs\ecowatch\video_detection\video_result_YOLO11s
Video processing completed.
Frames processed: 257
Frames with smoke detections: 230
Annotated video folder: C:\Projects\smoke_vehicle_detection\runs\ecowatch\video_detection\video_result


In [9]:
SAMPLE_IMAGE = DATASET / "test" / "images" / "negative_317173805_210226898147792_9136590978743834932_n_jpg.rf.d092438e4c304efa2ebd9ce300496479.jpg"
if SAMPLE_IMAGE.exists():
    best_model.predict(
        source=str(SAMPLE_IMAGE),
        conf=0.20,
        imgsz=IMAGE_SIZE,
        device=DEVICE,
        save=True,
        project=str(OUTPUT_ROOT),
        name="YOLO11s_preview",
        exist_ok=True,
    )
else:
    print("Update SAMPLE_IMAGE to an existing image path:", SAMPLE_IMAGE)


image 1/1 C:\Projects\smoke_vehicle_detection\EcoWatch-Final-YOLO\test\images\negative_317173805_210226898147792_9136590978743834932_n_jpg.rf.d092438e4c304efa2ebd9ce300496479.jpg: 640x640 (no detections), 98.6ms
Speed: 3.6ms preprocess, 98.6ms inference, 0.9ms postprocess per image at shape (1, 3, 640, 640)
Results saved to C:\Projects\smoke_vehicle_detection\runs\ecowatch\model_comparison\YOLO11s_preview


In [10]:

from pathlib import Path
from ultralytics import YOLO
import torch

PROJECT = Path(r"C:\Projects\smoke_vehicle_detection")

MODEL_PATH = (
    PROJECT / "runs" / "ecowatch" / "model_comparison"
    / "YOLO11s" / "weights" / "best.pt"
)

# Replace this with the path to your video.
VIDEO_PATH = PROJECT / 'test_video' / "test_video2.mp4"

OUTPUT_DIR = PROJECT / "runs" / "ecowatch" / "video_detection"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if not MODEL_PATH.exists():
    raise FileNotFoundError(f"Model not found: {MODEL_PATH}")

if not VIDEO_PATH.exists():
    raise FileNotFoundError(
        f"Video not found: {VIDEO_PATH}\n"
        "Change VIDEO_PATH to your actual video file."
    )

device = 0 if torch.cuda.is_available() else "cpu"
model = YOLO(str(MODEL_PATH))

results = model.predict(
    source=str(VIDEO_PATH),
    conf=0.25,
    imgsz=640,
    device=device,
    save=True,
    project=str(OUTPUT_DIR),
    name="video_result_YOLO11s2",
    exist_ok=True,
    stream=True,
    verbose=False
)

frame_count = 0
frames_with_smoke = 0

for result in results:
    frame_count += 1

    if result.boxes is not None and len(result.boxes) > 0:
        frames_with_smoke += 1

print("Video processing completed.")
print("Frames processed:", frame_count)
print("Frames with smoke detections:", frames_with_smoke)
print("Annotated video folder:", OUTPUT_DIR / "video_result")


Results saved to C:\Projects\smoke_vehicle_detection\runs\ecowatch\video_detection\video_result_YOLO11s2
Video processing completed.
Frames processed: 156
Frames with smoke detections: 8
Annotated video folder: C:\Projects\smoke_vehicle_detection\runs\ecowatch\video_detection\video_result
